# TrOCR Fine-Tuning — Kaggle GPU Training

Fine-tunes `microsoft/trocr-base-handwritten` on the Devanagari character data as the comparison model against the CRNN baseline.

Dataset: `sanskritipoudel/devnagari-preprocessed` (attach it). Code: repo branch `ml`.
Requires **GPU T4** and **Internet ON** (Settings panel) so the pretrained weights can download.
Run all cells top to bottom. Outputs land in `/kaggle/working/artifacts`.

In [ ]:
# 1. Confirm GPU is on
import torch
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4'
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code (branch: ml). transformers is preinstalled on Kaggle; add jiwer for eval.
import os
if not os.path.isdir('/kaggle/working/Devnagari_Handwriting_Recognition'):
    !git clone -b ml https://github.com/Sanskriti-poudel/Devnagari_Handwriting_Recognition.git /kaggle/working/Devnagari_Handwriting_Recognition
%cd /kaggle/working/Devnagari_Handwriting_Recognition
!git log --oneline -1
!pip install -q jiwer

In [ ]:
# 3. Locate the dataset root (the folder that directly contains train/) and point the loader at it.
import os, json

def find_data_root(base='/kaggle/input'):
    for dirpath, dirnames, _ in os.walk(base):
        if 'train' in dirnames and os.path.isdir(os.path.join(dirpath, 'train')):
            return dirpath
    raise FileNotFoundError(f'No folder containing train/ found under {base}')

DATA_ROOT = find_data_root()
os.environ['DEVNAGARI_DATA_ROOT'] = DATA_ROOT
print('DEVNAGARI_DATA_ROOT =', DATA_ROOT)
split = json.load(open('data/split_index.json'))
sample = os.path.join(DATA_ROOT, split['train'][0])
print('sample path:', sample, '-> exists:', os.path.exists(sample))

In [ ]:
# 3b. Validate the data path before spending GPU time (network-free).
!python models/trocr/smoke_test.py

In [ ]:
# 4. Fine-tune. batch_size=4 to fit trocr-base on a 16GB T4 (batch 8 OOMs).
#    Output is tee'd to artifacts/train.log so any crash is recoverable.
#    NOTE (2026-06-16 fixes): decoder_start_token_id now matches the pretrained
#    TrOCR (sep/2, was wrongly cls/0 -> empty generations -> 0% acc) and TrOCR
#    inputs are now inverted to dark-on-light to match the pretrained domain.
#    Epochs bumped 3->8 so the ViT encoder has time to adapt; watch val loss fall
#    (the old broken run sat ~9-15, i.e. near random).
#    Knobs: TROCR_BATCH_SIZE (drop to 2 if OOM), TROCR_EPOCHS, TROCR_LR,
#    TROCR_MODEL=microsoft/trocr-small-handwritten (faster), TROCR_MAX_TRAIN (0=all).
import os
os.environ['TROCR_BATCH_SIZE'] = '4'
os.environ['TROCR_EPOCHS'] = '8'
os.environ['TROCR_MAX_TRAIN'] = '0'
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
!python -u models/trocr/train.py 2>&1 | tee /kaggle/working/artifacts/train.log

In [ ]:
# 5. Evaluate on the held-out test split (Accuracy / CER / WER). Tee'd to eval.log.
!python -u models/trocr/evaluate.py --checkpoint models/trocr/checkpoints --device cuda 2>&1 | tee /kaggle/working/artifacts/eval.log

In [ ]:
# 6. Copy outputs to /kaggle/working/artifacts so they download with the version.
import shutil, os
os.makedirs('/kaggle/working/artifacts/trocr', exist_ok=True)
artifact_files = [
    'logs/trocr_training.csv', 'logs/trocr_eval.json',
    'logs/comparison.md', 'logs/comparison_metrics.png', 'logs/comparison_size.png',
    'logs/trocr_error_analysis.md', 'logs/trocr_confusion_pairs.csv',
    'logs/trocr_confusion_heatmap.png',
]
for f in artifact_files:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/artifacts/'); print('saved', f)
if os.path.isdir('models/trocr/checkpoints'):
    shutil.copytree('models/trocr/checkpoints', '/kaggle/working/artifacts/trocr/checkpoints', dirs_exist_ok=True)
    print('saved checkpoint dir')
print(os.listdir('/kaggle/working/artifacts'))

In [ ]:
# 5c. Populate the comparison table/plots (CRNN vs TrOCR) and TrOCR qualitative
#     error analysis — so this single run produces all the Phase-3 deliverables.
!python models/compare.py
!python models/error_analysis.py --model trocr --checkpoint models/trocr/checkpoints --device cuda

In [ ]:
# 6. Copy outputs to /kaggle/working/artifacts so they download with the version.
import shutil, os
os.makedirs('/kaggle/working/artifacts/trocr', exist_ok=True)
for f in ['logs/trocr_training.csv', 'logs/trocr_eval.json']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/artifacts/'); print('saved', f)
if os.path.isdir('models/trocr/checkpoints'):
    shutil.copytree('models/trocr/checkpoints', '/kaggle/working/artifacts/trocr/checkpoints', dirs_exist_ok=True)
    print('saved checkpoint dir')
print(os.listdir('/kaggle/working/artifacts'))